# 🧠 Orquestração de IA: SDD, TDD e Multi-Agentes

**Companion Notebook para o Artigo Completo sobre Orquestração Inteligente de Agentes**

Este notebook implementa na prática os conceitos apresentados no artigo:

- **SDD** (Specification-Driven Development) — Especificações formais com Pydantic
- **TDD** (Test-Driven Development) — Ciclo RED → GREEN → REFACTOR
- **Orquestração Multi-Agente** — Registro, roteamento e execução de agentes
- **Sistema de Hooks** — Eventos de ciclo de vida
- **Engenharia de Prompt** — Padrões comprovados (CoT, Few-shot, JSON mode)

> 💡 **Não requer API keys externas** — todos os agentes são simulados para demonstração.


In [ ]:
#@title Setup e Verificação do Ambiente
import sys, os, json, textwrap, warnings, time, random, typing, inspect, re
from datetime import datetime
from typing import List, Dict, Optional, Any, Callable, Tuple
from enum import Enum
from abc import ABC, abstractmethod
from collections import defaultdict

warnings.filterwarnings('ignore')

print(f"🐍 Python {sys.version}")
print(f"📅 Data: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

# Install dependencies
!pip install -q openai anthropic pydantic pytest duckduckgo_search rich 2>&1 | tail -1

# Verify installations
for pkg_name, pkg_import in [('pydantic', 'pydantic'), ('rich', 'rich')]:
    try:
        mod = __import__(pkg_import)
        ver = getattr(mod, '__version__', 'desconhecida')
        print(f"✅ {pkg_name} {ver}")
    except Exception as e:
        print(f"⚠️ {pkg_name}: {e}")

print("✅ Ambiente pronto!")


# 📋 SDD — Specification-Driven Development

O **SDD** é uma abordagem onde o desenvolvimento começa pela especificação formal dos
requisitos antes de qualquer implementação.

## Componentes do SDD Engine:

1. **Spec** — Modelo Pydantic que define: `id`, `title`, `description`,
   `acceptance_criteria`, `test_cases`, `dependencies`, `version`
2. **SpecRegistry** — Registro central que armazena e valida especificações
3. **SpecVerifier** — Verifica se os critérios de aceitação são testáveis

### Exemplo: 3 especificações para um sistema "OrquestradorMultiAgente"


In [ ]:
#@title SDD Engine: Especificação com Pydantic
from pydantic import BaseModel, Field
from datetime import datetime
from typing import List, Dict, Optional, Any


class Spec(BaseModel):
    """Uma especificação formal no estilo SDD."""
    id: str = Field(..., description="Identificador único da spec")
    title: str = Field(..., description="Título descritivo")
    description: str = Field("", description="Descrição detalhada")
    acceptance_criteria: List[str] = Field(default_factory=list,
                                           description="Critérios de aceitação")
    test_cases: List[Dict[str, Any]] = Field(default_factory=list,
                                             description="Casos de teste")
    dependencies: List[str] = Field(default_factory=list,
                                    description="Dependências de outras specs")
    version: str = Field("1.0.0", description="Versão da spec")
    created_at: datetime = Field(default_factory=datetime.now)

    def is_valid(self) -> bool:
        return bool(self.id and self.title and len(self.acceptance_criteria) > 0)

    def __str__(self) -> str:
        return f"[{self.id}] {self.title} (v{self.version})"


class SpecRegistry:
    """Registro central de especificações."""

    def __init__(self):
        self._specs: Dict[str, Spec] = {}

    def register(self, spec: Spec) -> None:
        if spec.id in self._specs:
            raise ValueError(f"Spec '{spec.id}' já registrada")
        errors = self.validate(spec)
        if errors:
            raise ValueError(f"Spec '{spec.id}' inválida: {', '.join(errors)}")
        self._specs[spec.id] = spec

    def get(self, spec_id: str) -> Optional[Spec]:
        return self._specs.get(spec_id)

    def list(self) -> List[Spec]:
        return list(self._specs.values())

    def validate(self, spec: Spec) -> List[str]:
        errors = []
        if not spec.id:
            errors.append("id é obrigatório")
        if not spec.title:
            errors.append("title é obrigatório")
        if not spec.acceptance_criteria:
            errors.append("pelo menos um acceptance_criteria é obrigatório")
        return errors

    def __len__(self) -> int:
        return len(self._specs)


class SpecVerifier:
    """Verifica se uma especificação é testável e completa."""

    @staticmethod
    def is_testable(criteria: str) -> bool:
        """Verifica se um critério de aceitação é testável."""
        testable_keywords = [
            'deve', 'should', 'must', 'shall', 'will',
            'expect', 'verify', 'validate', 'check', 'ensure',
            'retornar', 'permitir', 'suportar', 'aceitar',
        ]
        return any(kw in criteria.lower() for kw in testable_keywords)

    @classmethod
    def verify_spec(cls, spec: Spec) -> Dict[str, Any]:
        """Analisa a testabilidade de uma spec."""
        total = len(spec.acceptance_criteria)
        testable = sum(1 for c in spec.acceptance_criteria if cls.is_testable(c))
        return {
            "spec_id": spec.id,
            "title": spec.title,
            "total_criteria": total,
            "testable_criteria": testable,
            "testability_score": round(testable / max(total, 1), 2),
            "has_tests": len(spec.test_cases) > 0,
            "is_valid": spec.is_valid(),
            "dependencies": spec.dependencies,
            "version": spec.version,
        }


# ===== EXEMPLO: 3 Specs para "OrquestradorMultiAgente" =====
registry = SpecRegistry()

spec_agentes = Spec(
    id="SPEC-001",
    title="Registro de Agentes",
    description="O orquestrador deve permitir registro dinâmico de agentes com capacidades.",
    acceptance_criteria=[
        "Deve permitir registrar um agente com nome único",
        "Deve validar capacidades mínimas do agente",
        "Deve retornar erro ao registrar agente duplicado",
        "Deve listar todos os agentes registrados",
    ],
    test_cases=[
        {"name": "test_register_valid_agent",
         "input": {"name": "Agent1", "capabilities": ["search"]},
         "expected": "success"},
        {"name": "test_register_duplicate",
         "input": {"name": "Agent1", "capabilities": ["search"]},
         "expected": "error"},
    ],
)

spec_roteamento = Spec(
    id="SPEC-002",
    title="Roteamento de Tarefas",
    description="Orquestrador deve rotear tarefas ao agente mais adequado por capacidade.",
    acceptance_criteria=[
        "Deve selecionar agente com maior compatibilidade de capacidades",
        "Deve retornar erro quando nenhum agente atende às capacidades requeridas",
        "Deve registrar histórico de execução para auditoria",
    ],
    dependencies=["SPEC-001"],
)

spec_hooks = Spec(
    id="SPEC-003",
    title="Sistema de Hooks",
    description="Orquestrador deve suportar hooks de ciclo de vida durante execução.",
    acceptance_criteria=[
        "Deve disparar hook on_task_start antes de executar tarefa",
        "Deve disparar hook on_task_complete após execução bem-sucedida",
        "Deve disparar hook on_error quando tarefa falha",
        "Deve acumular métricas de tempo de execução por agente",
    ],
    dependencies=["SPEC-001", "SPEC-002"],
)

for spec in [spec_agentes, spec_roteamento, spec_hooks]:
    registry.register(spec)

verifier = SpecVerifier()

print("=" * 70)
print("📋 SDD ENGINE — ESPECIFICAÇÕES DO ORQUESTRADOR MULTI-AGENTE")
print("=" * 70)

for s in registry.list():
    report = verifier.verify_spec(s)
    print(f"\n📌 {s}")
    print(f"   Descrição: {s.description}")
    print(f"   Critérios: {report['total_criteria']} total, "
          f"{report['testable_criteria']} testáveis")
    print(f"   Testabilidade: {report['testability_score']:.0%}")
    print(f"   Testes: {'✅' if report['has_tests'] else '⚠️'} "
          f"{'Definidos' if report['has_tests'] else 'Pendentes'}")
    print(f"   Dependências: {report['dependencies'] if report['dependencies'] else 'Nenhuma'}")

print(f"\n📊 Total de specs registradas: {len(registry)}")
print("✅ SDD Engine carregado com sucesso!")


# 🔄 TDD — Test-Driven Development

O ciclo **RED → GREEN → REFACTOR** é a base do TDD:

1. **🔴 RED** — Escreva um teste que *falha* (o código ainda não existe)
2. **🟢 GREEN** — Implemente o código mínimo para *passar* o teste
3. **🔵 REFACTOR** — Melhore o código *mantendo os testes verdes*

### Componentes:
- **TestRunner** — Descobre e executa testes, reporta resultados
- **CoverageTracker** — Rastreia linhas cobertas por testes (simulado)
- **Exemplo prático**: Calculadora com operações básicas


In [ ]:
#@title TDD Engine: Ciclo RED → GREEN → REFACTOR
import time
from typing import List, Dict, Any, Callable


class TestResult:
    """Resultado de um teste individual."""
    def __init__(self, name: str, passed: bool, reason: str = ""):
        self.name = name
        self.passed = passed
        self.reason = reason
        self.timestamp = time.time()


class TestRunner:
    """Executor de testes com suporte a fases TDD."""

    def __init__(self):
        self.tests: List[tuple] = []
        self.results: List[TestResult] = []

    def add_test(self, test_fn: Callable, name: str = None):
        self.tests.append((test_fn, name or test_fn.__name__))

    def run(self) -> List[TestResult]:
        self.results = []
        for test_fn, name in self.tests:
            try:
                test_fn()
                self.results.append(TestResult(name, True))
            except AssertionError as e:
                self.results.append(TestResult(name, False,
                                               f"AssertionError: {e}"))
            except NameError as e:
                self.results.append(TestResult(name, False,
                                               f"NameError (RED): {e}"))
            except Exception as e:
                self.results.append(TestResult(name, False,
                                               f"{type(e).__name__}: {e}"))
        return self.results

    def summary(self) -> Dict[str, Any]:
        total = len(self.results)
        passed = sum(1 for r in self.results if r.passed)
        return {"total": total, "passed": passed, "failed": total - passed}


class CoverageTracker:
    """Rastreador simulado de cobertura de código."""
    def __init__(self):
        self.covered: set = set()
        self.total: set = set()

    def add_line(self, func: str, line: int):
        self.total.add((func, line))

    def mark_covered(self, func: str, line: int):
        self.covered.add((func, line))

    def report(self) -> Dict[str, Any]:
        total = len(self.total)
        covered = len(self.covered)
        pct = (covered / total * 100) if total > 0 else 0.0
        return {"total": total, "covered": covered, "coverage_pct": round(pct, 1)}


# ===== DEMONSTRAÇÃO: RED → GREEN → REFACTOR =====
print("=" * 60)
print("🔄 CICLO TDD: RED → GREEN → REFACTOR")
print("=" * 60)

runner = TestRunner()

# --- RED PHASE: Write tests that WILL fail (Calculator doesn't exist yet) ---
runner.phase = "red"
print("\n🔴 FASE RED: Escrevendo testes que falham...")

# NOTE: Calculator is NOT defined yet — these tests will raise NameError
def test_add():
    calc = Calculator()
    assert calc.add(2, 3) == 5

def test_subtract():
    calc = Calculator()
    assert calc.subtract(5, 3) == 2

def test_multiply():
    calc = Calculator()
    assert calc.multiply(2, 3) == 6

def test_divide():
    calc = Calculator()
    assert calc.divide(6, 3) == 2

def test_divide_by_zero():
    calc = Calculator()
    try:
        calc.divide(1, 0)
        assert False, "Deveria levantar ValueError"
    except ValueError:
        pass

for fn in [test_add, test_subtract, test_multiply, test_divide, test_divide_by_zero]:
    runner.add_test(fn)

red_results = runner.run()
red_fail = sum(1 for r in red_results if not r.passed)
print(f"  Total: {len(red_results)} | Passed: {len(red_results)-red_fail} | Failed: {red_fail}")
for r in red_results:
    status = "🔴 FAIL" if not r.passed else "✅ PASS"
    print(f"  {status} | {r.name}: {r.reason if not r.passed else 'OK'}")

# --- GREEN PHASE: Implement minimal code ---
print("\n🟢 FASE GREEN: Implementando código mínimo...")

class Calculator:
    """Calculadora simples — implementação mínima."""
    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b

    def multiply(self, a, b):
        return a * b

    def divide(self, a, b):
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

green_results = runner.run()
green_pass = sum(1 for r in green_results if r.passed)
print(f"  Total: {len(green_results)} | Passed: {green_pass} | Failed: {len(green_results)-green_pass}")
for r in green_results:
    status = "🟢 PASS" if r.passed else "❌ FAIL"
    print(f"  {status} | {r.name}")

# --- REFACTOR PHASE: Improve without breaking tests ---
print("\n🔵 FASE REFACTOR: Refatorando com type hints e validações...")

class CalculatorRefactored:
    """Calculadora refatorada com type hints e documentação."""

    def add(self, a: float, b: float) -> float:
        """Soma dois números."""
        return a + b

    def subtract(self, a: float, b: float) -> float:
        """Subtrai b de a."""
        return a - b

    def multiply(self, a: float, b: float) -> float:
        """Multiplica dois números."""
        return a * b

    def divide(self, a: float, b: float) -> float:
        """Divide a por b. Levanta ValueError se b for zero."""
        if b == 0:
            raise ValueError("Cannot divide by zero")
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            raise TypeError("Argumentos devem ser números")
        return a / b

# Verify refactored version still passes tests (rebind Calculator)
Calculator = CalculatorRefactored
refactor_results = runner.run()
refactor_pass = sum(1 for r in refactor_results if r.passed)
print(f"  Total: {len(refactor_results)} | Passed: {refactor_pass} | "
      f"Failed: {len(refactor_results)-refactor_pass}")
print(f"  ✅ Refatoração completa — todos os testes ainda passam!")

print("\n✅ CICLO TDD CONCLUÍDO COM SUCESSO!")


# 🤖 Orquestração Multi-Agente

Arquitetura de um orquestrador capaz de coordenar múltiplos agentes especializados.

## Componentes:

1. **BaseAgent** — Classe abstrata com `name`, `role`, `capabilities`, `max_retries`
2. **Orchestrator** — Registra agentes, roteia tarefas por capacidade, coleta métricas
3. **Agentes Concretos**:
   - 🔬 **ResearcherAgent** — Pesquisa e análise
   - ✍️ **WriterAgent** — Escrita e sumarização
   - ✅ **ReviewerAgent** — Revisão e validação

### Roteamento Inteligente:
- Matching por score de capacidades
- Retry com tolerância a falhas
- Histórico completo de execução


In [ ]:
#@title Orquestrador Multi-Agente
import time
import random
from abc import ABC, abstractmethod
from typing import Dict, List, Optional, Any, Callable
from collections import defaultdict
from datetime import datetime


# ===== TASK =====
class Task:
    """Representa uma tarefa a ser executada por um agente."""

    def __init__(self, id: str, description: str,
                 required_capabilities: List[str],
                 context: Optional[Dict] = None):
        self.id = id
        self.description = description
        self.required_capabilities = required_capabilities
        self.context = context or {}
        self.status = "pending"
        self.result = None
        self.agent_id = None
        self.started_at = None
        self.completed_at = None
        self.error = None

    @property
    def duration(self) -> Optional[float]:
        if self.started_at and self.completed_at:
            return self.completed_at - self.started_at
        return None

    def __repr__(self) -> str:
        return f"Task({self.id}, status={self.status})"


# ===== BASE AGENT =====
class BaseAgent(ABC):
    """Agente abstrato base para o orquestrador multi-agente."""

    def __init__(self, name: str, role: str, capabilities: List[str],
                 max_retries: int = 3):
        self.name = name
        self.role = role
        self.capabilities = capabilities
        self.max_retries = max_retries
        self.execution_count = 0

    @abstractmethod
    def execute(self, task: Task) -> Dict[str, Any]:
        """Executa uma tarefa e retorna o resultado."""
        ...

    def can_handle(self, capability: str) -> bool:
        return capability in self.capabilities

    def __repr__(self) -> str:
        return f"{self.role}({self.name})"


# ===== CONCRETE AGENTS =====
class ResearcherAgent(BaseAgent):
    """Agente especializado em pesquisa e análise."""

    def __init__(self):
        super().__init__("Researcher-1", "Pesquisador",
                         ["search", "analyze", "research", "collect"])

    def execute(self, task: Task) -> Dict[str, Any]:
        time.sleep(random.uniform(0.05, 0.15))
        topics = task.context.get("topics",
                                  ["orquestração", "multi-agente", "SDD", "TDD"])
        self.execution_count += 1
        return {
            "findings": [f"Resultado da pesquisa sobre {t}" for t in topics],
            "sources": len(topics),
            "confidence": round(random.uniform(0.75, 0.95), 2),
        }


class WriterAgent(BaseAgent):
    """Agente especializado em escrita e sumarização."""

    def __init__(self):
        super().__init__("Writer-1", "Escritor",
                         ["write", "summarize", "format", "document"])

    def execute(self, task: Task) -> Dict[str, Any]:
        time.sleep(random.uniform(0.05, 0.12))
        content_length = task.context.get("content_length", 300)
        self.execution_count += 1
        return {
            "content": f"Conteúdo gerado sobre {task.description[:30]}...",
            "word_count": content_length + random.randint(-50, 50),
            "format": "markdown",
        }


class ReviewerAgent(BaseAgent):
    """Agente especializado em revisão e validação."""

    def __init__(self):
        super().__init__("Reviewer-1", "Revisor",
                         ["review", "validate", "check", "quality"])

    def execute(self, task: Task) -> Dict[str, Any]:
        time.sleep(random.uniform(0.03, 0.10))
        self.execution_count += 1
        is_approved = random.random() > 0.3
        return {
            "approved": is_approved,
            "score": round(random.uniform(0.6, 1.0), 2),
            "issues": [] if is_approved else random.sample(
                ["clarity", "accuracy", "completeness", "format"],
                k=random.randint(1, 2)),
            "suggestions": "Nenhuma" if is_approved else "Revisar seções indicadas",
        }


# ===== ORCHESTRATOR =====
class Orchestrator:
    """Orquestrador multi-agente com roteamento inteligente e hooks."""

    def __init__(self):
        self.agents: Dict[str, BaseAgent] = {}
        self.task_history: List[Dict] = []
        self.metrics: Dict[str, Dict] = defaultdict(lambda: {
            "executions": 0, "total_time": 0.0, "avg_time": 0.0, "errors": 0
        })
        self._hooks: Dict[str, List[Callable]] = defaultdict(list)

    # --- Hook Management ---
    def on(self, event: str, callback: Callable):
        """Registra um callback para um evento."""
        self._hooks[event].append(callback)

    def _trigger(self, event: str, **kwargs):
        """Dispara todos os callbacks registrados para um evento."""
        for cb in self._hooks.get(event, []):
            try:
                cb(**kwargs)
            except Exception:
                pass

    # --- Agent Management ---
    def register_agent(self, agent: BaseAgent):
        """Registra um agente no orquestrador."""
        if agent.name in self.agents:
            raise ValueError(f"Agente '{agent.name}' já registrado")
        if not agent.capabilities:
            raise ValueError(f"Agente '{agent.name}' deve ter ao menos uma capacidade")
        self.agents[agent.name] = agent
        self._trigger("on_agent_register", agent=agent)

    def get_agent(self, name: str) -> Optional[BaseAgent]:
        return self.agents.get(name)

    def list_agents(self) -> List[BaseAgent]:
        return list(self.agents.values())

    # --- Task Routing ---
    def find_best_agent(self, required_capabilities: List[str]) -> Optional[BaseAgent]:
        """Encontra o agente com maior compatibilidade de capacidades."""
        best_agent = None
        best_score = -1
        for agent in self.agents.values():
            score = sum(1 for cap in required_capabilities if cap in agent.capabilities)
            if score > best_score:
                best_score = score
                best_agent = agent
        return best_agent if best_score > 0 else None

    def execute_task(self, task: Task) -> Task:
        """Executa uma tarefa roteando para o melhor agente disponível."""
        self._trigger("on_task_start", task=task)
        task.started_at = time.time()

        agent = self.find_best_agent(task.required_capabilities)
        if agent is None:
            task.status = "failed"
            task.error = "Nenhum agente disponível com as capacidades requeridas"
            task.completed_at = time.time()
            self._trigger("on_error", task=task, error=task.error)
            return task

        task.agent_id = agent.name

        for attempt in range(agent.max_retries):
            try:
                result = agent.execute(task)
                task.status = "completed"
                task.result = result
                task.completed_at = time.time()

                duration = task.duration or 0
                m = self.metrics[agent.name]
                m["executions"] += 1
                m["total_time"] += duration
                m["avg_time"] = m["total_time"] / m["executions"]

                self.task_history.append({
                    "task_id": task.id,
                    "agent": agent.name,
                    "status": "completed",
                    "duration": round(duration, 3),
                    "timestamp": datetime.now().isoformat(),
                })

                self._trigger("on_task_complete", task=task, agent=agent)
                return task

            except Exception as e:
                if attempt == agent.max_retries - 1:
                    task.status = "failed"
                    task.error = str(e)
                    task.completed_at = time.time()
                    self.metrics[agent.name]["errors"] += 1
                    self._trigger("on_error", task=task, error=str(e))
                continue

        return task

    # --- Metrics ---
    def get_summary(self) -> Dict[str, Any]:
        """Retorna sumário de execução do orquestrador."""
        total_tasks = len(self.task_history)
        completed = sum(1 for t in self.task_history if t["status"] == "completed")
        failed = total_tasks - completed
        total_time = sum(t["duration"] for t in self.task_history)
        return {
            "agents": len(self.agents),
            "total_tasks": total_tasks,
            "completed": completed,
            "failed": failed,
            "total_time": round(total_time, 3),
            "avg_time_per_task": round(total_time / max(total_tasks, 1), 3),
            "per_agent": dict(self.metrics),
        }


# ===== QUICK SANITY CHECK =====
print("=" * 60)
print("✅ Módulos do Orquestrador carregados com sucesso!")
print("   • Task, BaseAgent, ResearcherAgent, WriterAgent, ReviewerAgent")
print("   • Orchestrator com hooks e roteamento")
print("=" * 60)


# 🔌 Sistema de Hooks — Eventos do Ciclo de Vida

Hooks permitem estender o comportamento do orquestrador sem modificar seu código.

## Eventos Suportados:
- `on_task_start` — Antes de cada tarefa
- `on_task_complete` — Após tarefa bem-sucedida
- `on_error` — Quando uma tarefa falha
- `on_agent_register` — Quando um novo agente é registrado

## Hooks Implementados:
- **LoggingHook** — Registra eventos em log textual
- **MetricsHook** — Acumula métricas de tempo e performance


In [ ]:
#@title Sistema de Hooks: Eventos do Ciclo de Vida
from typing import Callable, Dict, List, Any
from collections import defaultdict
from datetime import datetime


class HookManager:
    """Gerenciador de hooks com registro, remoção e disparo."""

    def __init__(self):
        self._hooks: Dict[str, List[Callable]] = defaultdict(list)

    def register(self, event: str, hook_fn: Callable):
        if not callable(hook_fn):
            raise TypeError("hook_fn deve ser callable")
        self._hooks[event].append(hook_fn)

    def unregister(self, event: str, hook_fn: Callable):
        if event in self._hooks:
            self._hooks[event] = [h for h in self._hooks[event] if h != hook_fn]

    def trigger(self, event: str, **kwargs):
        for hook_fn in self._hooks.get(event, []):
            try:
                hook_fn(**kwargs)
            except Exception as e:
                print(f"  ⚠️ Hook error: {e}")

    def registered_events(self) -> List[str]:
        return list(self._hooks.keys())

    def hook_count(self, event: str) -> int:
        return len(self._hooks.get(event, []))


class LoggingHook:
    """Hook de logging que registra eventos em uma lista."""

    def __init__(self):
        self.logs: List[str] = []

    def on_task_start(self, **kwargs):
        task = kwargs.get('task')
        self.logs.append(
            f"[{datetime.now().strftime('%H:%M:%S')}] ▶️ Iniciando: "
            f"{task.id} - {task.description[:60]}")

    def on_task_complete(self, **kwargs):
        task = kwargs.get('task')
        agent = kwargs.get('agent')
        dur = round(task.duration, 3) if task.duration else 0
        self.logs.append(
            f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Concluído: "
            f"{task.id} por {agent.name} ({dur}s)")

    def on_error(self, **kwargs):
        task = kwargs.get('task')
        error = kwargs.get('error')
        self.logs.append(
            f"[{datetime.now().strftime('%H:%M:%S')}] ❌ Erro: "
            f"{task.id} - {error}")

    def on_agent_register(self, **kwargs):
        agent = kwargs.get('agent')
        caps = ', '.join(agent.capabilities)
        self.logs.append(
            f"[{datetime.now().strftime('%H:%M:%S')}] 🤖 Agente: "
            f"{agent.name} ({agent.role}) - [{caps}]")

    def show_logs(self):
        print("\n".join(self.logs))


class MetricsHook:
    """Hook de métricas que acumula estatísticas de execução."""

    def __init__(self):
        self.metrics: Dict[str, Any] = {
            "tasks": [],
            "agent_times": defaultdict(list),
            "total_duration": 0.0,
            "errors": [],
        }

    def on_task_complete(self, **kwargs):
        task = kwargs.get('task')
        agent = kwargs.get('agent')
        duration = task.duration or 0
        self.metrics["tasks"].append({
            "task_id": task.id,
            "agent": agent.name,
            "duration": round(duration, 3),
        })
        self.metrics["agent_times"][agent.name].append(duration)
        self.metrics["total_duration"] += duration

    def on_error(self, **kwargs):
        task = kwargs.get('task')
        self.metrics["errors"].append({"task_id": task.id, "error": task.error})

    def summary(self) -> Dict[str, Any]:
        total = len(self.metrics["tasks"])
        errors = len(self.metrics["errors"])
        agent_stats = {}
        for agent, times in self.metrics["agent_times"].items():
            agent_stats[agent] = {
                "executions": len(times),
                "total_time": round(sum(times), 3),
                "avg_time": round(sum(times) / max(len(times), 1), 3),
            }
        return {
            "total_tasks": total,
            "completed": total - errors,
            "errors": errors,
            "total_time": round(self.metrics["total_duration"], 3),
            "avg_time": round(self.metrics["total_duration"] / max(total, 1), 3),
            "per_agent": agent_stats,
        }


# ===== DEMO: Integração com Orquestrador =====
print("=" * 60)
print("🔌 DEMO: Hooks no Orquestrador")
print("=" * 60)

orc = Orchestrator()
logging_hook = LoggingHook()
metrics_hook = MetricsHook()

# Register hooks via Orchestrator's event system
orc.on("on_agent_register", logging_hook.on_agent_register)
orc.on("on_task_start", logging_hook.on_task_start)
orc.on("on_task_complete", logging_hook.on_task_complete)
orc.on("on_task_complete", metrics_hook.on_task_complete)
orc.on("on_error", logging_hook.on_error)
orc.on("on_error", metrics_hook.on_error)

# Register agents
for agent in [ResearcherAgent(), WriterAgent(), ReviewerAgent()]:
    orc.register_agent(agent)

# Execute sample tasks
tasks_data = [
    Task("TASK-001", "Pesquisar sobre orquestração de IA",
         ["search", "research"]),
    Task("TASK-002", "Escrever resumo dos resultados",
         ["write", "summarize"]),
    Task("TASK-003", "Revisar o conteúdo gerado",
         ["review", "validate"]),
    Task("TASK-004", "Pesquisar técnicas avançadas",
         ["search", "analyze"]),
]

print("\n📋 Executando tarefas com hooks...\n")
for t in tasks_data:
    orc.execute_task(t)

print("\n📋 Logs de Execução:")
print("-" * 40)
logging_hook.show_logs()

print("\n📊 Métricas Coletadas:")
print("-" * 40)
hook_summary = metrics_hook.summary()
for agent, stats in hook_summary["per_agent"].items():
    print(f"   🤖 {agent}: {stats['executions']} execuções, "
          f"média {stats['avg_time']:.3f}s")

print(f"\n   📈 Total: {hook_summary['total_tasks']} tarefas em "
      f"{hook_summary['total_time']:.3f}s")
print(f"   ⚠️  Erros: {hook_summary['errors']}")
print("\n✅ Sistema de hooks demonstrado com sucesso!")


# 🧠 Padrões de Engenharia de Prompt

Técnicas comprovadas para construir prompts eficazes para agentes de IA.

## Padrões Implementados:

1. **Chain-of-Thought (CoT)** — Raciocínio passo a passo
2. **Saída Estruturada (JSON Mode)** — Schema de saída explícito
3. **Few-Shot Learning** — Exemplos contextuais antes da consulta
4. **System Prompt Builder** — Template com papel, contexto e restrições
5. **Agent Prompt Builder** — Prompt específico para cada agente do orquestrador


In [ ]:
#@title Padrões de Engenharia de Prompt
import json
from typing import List, Dict, Any, Optional, Tuple


class PromptBuilder:
    """Construtor de templates de prompt com padrões comprovados."""

    @staticmethod
    def chain_of_thought(question: str,
                         reasoning_steps: Optional[List[str]] = None) -> str:
        """Chain-of-Thought: raciocínio passo a passo."""
        prompt = f"Pergunta: {question}\n\nVamos resolver passo a passo:\n"
        if reasoning_steps:
            for i, step in enumerate(reasoning_steps, 1):
                prompt += f"{i}. {step}\n"
        prompt += "\nConclusão:"
        return prompt

    @staticmethod
    def structured_output(system_prompt: str, output_schema: Dict) -> str:
        """Prompt com saída estruturada em JSON."""
        schema_str = json.dumps(output_schema, indent=2, ensure_ascii=False)
        return f"""{system_prompt}

IMPORTANTE: Responda APENAS com JSON válido neste formato:
{schema_str}

Não inclua texto antes ou depois do JSON."""

    @staticmethod
    def few_shot(examples: List[Tuple[str, str]], query: str) -> str:
        """Few-shot learning com exemplos."""
        prompt = "Exemplos:\n\n"
        for i, (inp, out) in enumerate(examples, 1):
            prompt += f"Exemplo {i}:\nEntrada: {inp}\nSaída: {out}\n\n"
        prompt += f"Agora resolva:\nEntrada: {query}\nSaída:"
        return prompt

    @staticmethod
    def system_prompt(role: str, context: str,
                      constraints: List[str],
                      output_format: Optional[str] = None) -> str:
        """System prompt builder com papel, contexto e restrições."""
        prompt = f"""# Papel
Você é um {role}.

# Contexto
{context}

# Restrições"""
        for c in constraints:
            prompt += f"\n- {c}"
        if output_format:
            prompt += f"\n\n# Formato de Saída\n{output_format}"
        return prompt

    @staticmethod
    def agent_system_prompt(agent_name: str, agent_role: str,
                            capabilities: List[str],
                            guidelines: List[str]) -> str:
        """System prompt específico para um agente do orquestrador."""
        caps_str = ", ".join(capabilities)
        guidelines_str = "\n".join(f"- {g}" for g in guidelines)
        return f"""# Identidade
Você é {agent_name}, um {agent_role} no sistema de orquestração multi-agente.

# Capacidades
Você pode executar tarefas relacionadas a: {caps_str}

# Diretrizes de Execução
{guidelines_str}

# Formato de Resposta
Sempre retorne um dicionário JSON com:
- "status": "success" ou "error"
- "result": o resultado da sua execução
- "confidence": nível de confiança (0.0 a 1.0)"""


# ===== DEMONSTRAÇÃO DOS PADRÕES =====
print("=" * 70)
print("🧠 PADRÕES DE ENGENHARIA DE PROMPT")
print("=" * 70)

# 1. Chain-of-Thought
print("\n" + "─" * 70)
print("📐 1. CHAIN-OF-THOUGHT (Cadeia de Raciocínio)")
print("─" * 70)
cot_prompt = PromptBuilder.chain_of_thought(
    "Como implementar um orquestrador multi-agente com SDD e TDD?",
    ["Definir especificações (SDD) para cada componente",
     "Escrever testes (TDD) que validam o comportamento",
     "Implementar código mínimo para passar nos testes",
     "Refatorar mantendo os testes verdes",
     "Integrar os componentes no orquestrador final"],
)
print(cot_prompt)

# 2. Structured Output
print("\n" + "─" * 70)
print("📐 2. SAÍDA ESTRUTURADA (JSON Mode)")
print("─" * 70)
schema = {
    "agent": "string (nome do agente)",
    "task_id": "string",
    "result": {
        "status": "string (success/error)",
        "data": "object (opcional)",
        "error": "string (opcional)",
    },
    "execution_time_ms": "number",
    "confidence": "number (0-1)",
}
structured_prompt = PromptBuilder.structured_output(
    "Você é um agente de execução de tarefas.", schema)
print(structured_prompt[:500] + "\n...")

# 3. Few-Shot
print("\n" + "─" * 70)
print("📐 3. FEW-SHOT LEARNING")
print("─" * 70)
few_shot_prompt = PromptBuilder.few_shot(
    [("Extraia os verbos: O sistema valida dados.",
      '{"verbos": ["valida"]}'),
     ("Extraia os verbos: O agente pesquisa, analisa e escreve.",
      '{"verbos": ["pesquisa", "analisa", "escreve"]}')],
    "Extraia os verbos: O orquestrador registra agentes, roteia tarefas e coleta métricas.",
)
print(few_shot_prompt)

# 4. System Prompt
print("\n" + "─" * 70)
print("📐 4. SYSTEM PROMPT BUILDER")
print("─" * 70)
agent_prompt = PromptBuilder.agent_system_prompt(
    "Researcher-1", "Pesquisador Especialista",
    ["search", "analyze", "research", "collect"],
    ["Sempre valide as fontes de informação",
     "Retorne apenas resultados com confiança > 0.7",
     "Inclua referências quando possível"],
)
print(agent_prompt)

print("\n" + "=" * 70)
print("✅ Padrões de prompt demonstrados com sucesso!")
print("=" * 70)


# 🚀 Pipeline Completo: SDD → TDD → Orquestração

Este pipeline integra todos os conceitos em uma única demonstração:

## Fases:
1. **📋 SDD** — Criação e validação de 3 specs para o sistema
2. **🟥 TDD RED** — 5 testes para o orquestrador
3. **🟩 TDD GREEN** — Implementação já existente (testes passam)
4. **🔧 Orquestração** — 5 tarefas executadas com hooks e métricas
5. **🧠 Prompt Engineering** — System prompts gerados para cada agente


In [ ]:
#@title Pipeline Completo: SDD → TDD → Orquestração com Hooks
import time
from datetime import datetime

print("=" * 70)
print("🚀 PIPELINE COMPLETO: SDD → TDD → ORQUESTRAÇÃO MULTI-AGENTE")
print("=" * 70)
print()

pipeline_start = time.time()

# ===== PHASE 1: SDD =====
print("📋 FASE 1: SDD — Definição de Especificações")
print("-" * 50)
phase_start = time.time()

pipeline_registry = SpecRegistry()

spec_pesquisa = Spec(
    id="SPEC-101",
    title="Pesquisa e Coleta",
    description="Agente pesquisador coleta informações sobre tópicos definidos.",
    acceptance_criteria=[
        "Deve aceitar tópicos de pesquisa como entrada",
        "Deve retornar uma lista de descobertas",
        "Deve incluir nível de confiança nos resultados",
    ],
    test_cases=[{"input": {"topics": ["IA"]},
                 "expected_fields": ["findings", "confidence"]}],
)

spec_conteudo = Spec(
    id="SPEC-102",
    title="Geração de Conteúdo",
    description="Agente escritor gera conteúdo a partir de dados pesquisados.",
    acceptance_criteria=[
        "Deve gerar conteúdo em markdown",
        "Deve respeitar o tamanho solicitado",
        "Deve referenciar as fontes da pesquisa",
    ],
    dependencies=["SPEC-101"],
)

spec_revisao = Spec(
    id="SPEC-103",
    title="Revisão e Validação",
    description="Agente revisor valida o conteúdo gerado.",
    acceptance_criteria=[
        "Deve verificar clareza e precisão",
        "Deve retornar score de qualidade",
        "Deve listar issues encontradas",
    ],
    dependencies=["SPEC-101", "SPEC-102"],
)

for s in [spec_pesquisa, spec_conteudo, spec_revisao]:
    pipeline_registry.register(s)

phase_1_time = time.time() - phase_start
print(f"   ✅ {len(pipeline_registry)} specs registradas e validadas")
print(f"   ⏱️  {phase_1_time:.3f}s")
print()

# ===== PHASE 2: TDD RED =====
print("🟥 FASE 2: TDD RED — Escrevendo Testes para o Orquestrador")
print("-" * 50)
phase_start = time.time()

tdd_runner = TestRunner()

def test_orchestrator_register_agent():
    orch = Orchestrator()
    orch.register_agent(ResearcherAgent())
    assert len(orch.list_agents()) == 1

def test_orchestrator_find_best_agent():
    orch = Orchestrator()
    orch.register_agent(ResearcherAgent())
    orch.register_agent(WriterAgent())
    best = orch.find_best_agent(["search", "analyze"])
    assert best is not None
    assert best.name == "Researcher-1"

def test_orchestrator_execute_task():
    orch = Orchestrator()
    orch.register_agent(ResearcherAgent())
    task = Task("T-TEST", "Pesquisar IA", ["search"])
    result = orch.execute_task(task)
    assert result.status == "completed"
    assert result.result is not None

def test_orchestrator_no_agent_found():
    orch = Orchestrator()
    task = Task("T-FAIL", "Tarefa impossível", ["fly", "teleport"])
    result = orch.execute_task(task)
    assert result.status == "failed"

def test_orchestrator_duplicate_agent():
    orch = Orchestrator()
    orch.register_agent(ResearcherAgent())
    try:
        orch.register_agent(ResearcherAgent())
        assert False, "Deveria levantar ValueError"
    except ValueError:
        pass

for fn in [test_orchestrator_register_agent,
           test_orchestrator_find_best_agent,
           test_orchestrator_execute_task,
           test_orchestrator_no_agent_found,
           test_orchestrator_duplicate_agent]:
    tdd_runner.add_test(fn)

red_results = tdd_runner.run()
red_passed = sum(1 for r in red_results if r.passed)
phase_2_time = time.time() - phase_start
print(f"   📊 Testes: {len(red_results)} total, {red_passed} passaram")
for r in red_results:
    status = "✅" if r.passed else "❌"
    print(f"   {status} {r.name}")
print(f"   ⏱️  {phase_2_time:.3f}s")
print()

# ===== PHASE 3: TDD GREEN / REFACTOR =====
print("🟩 FASE 3: TDD GREEN — Código já implementado e testado")
print("-" * 50)
print("   ✅ Classes base (Task, BaseAgent, Orchestrator) implementadas")
print(f"   ✅ {red_passed}/{len(red_results)} testes passam na fase GREEN")
print()

# ===== PHASE 4: Orquestração com Hooks =====
print("🔧 FASE 4: Orquestração com Hooks e Métricas")
print("-" * 50)
phase_start = time.time()

pipeline_orc = Orchestrator()
pipeline_logging = LoggingHook()
pipeline_metrics = MetricsHook()

# Register all hooks
for event in ["on_agent_register", "on_task_start",
              "on_task_complete", "on_error"]:
    pipeline_orc.on(event, getattr(pipeline_logging,
                                   event.replace("on_", "on_")))
pipeline_orc.on("on_task_complete", pipeline_metrics.on_task_complete)
pipeline_orc.on("on_error", pipeline_metrics.on_error)

# Register agents
for agent in [ResearcherAgent(), WriterAgent(), ReviewerAgent()]:
    pipeline_orc.register_agent(agent)

# Pipeline tasks
pipeline_tasks = [
    Task("PL-001", "Pesquisar fundamentos de orquestração de IA",
         ["search", "research"],
         {"topics": ["orquestração", "multi-agente", "SDD"]}),
    Task("PL-002", "Analisar resultados da pesquisa",
         ["analyze", "research"], {"depth": "comprehensive"}),
    Task("PL-003", "Escrever artigo sobre orquestração",
         ["write", "document"], {"content_length": 500}),
    Task("PL-004", "Revisar qualidade do artigo",
         ["review", "quality"], {"strict": True}),
    Task("PL-005", "Gerar resumo executivo",
         ["summarize", "write"], {"max_words": 100}),
]

print(f"   📋 {len(pipeline_tasks)} tarefas na fila")
for task in pipeline_tasks:
    pipeline_orc.execute_task(task)
    icon = "✅" if task.status == "completed" else "❌"
    dur = f"{task.duration:.3f}s" if task.duration else "N/A"
    print(f"   {icon} {task.id} → {task.agent_id} ({dur})")

phase_4_time = time.time() - phase_start
summary = pipeline_orc.get_summary()
print(f"   ⏱️  {phase_4_time:.3f}s")
print()

# ===== PHASE 5: Prompt Engineering Application =====
print("🧠 FASE 5: Aplicação de Padrões de Prompt")
print("-" * 50)
phase_start = time.time()

prompts = {}
for agent in pipeline_orc.list_agents():
    prompt = PromptBuilder.agent_system_prompt(
        agent.name, agent.role, agent.capabilities,
        ["Siga as diretrizes de qualidade",
         "Registre todas as execuções para auditoria"],
    )
    prompts[agent.name] = prompt

phase_5_time = time.time() - phase_start
print(f"   ✅ System prompts gerados para {len(prompts)} agentes")
for name, prompt in prompts.items():
    preview = prompt.split("\n")[0]
    print(f"   🤖 {name}: {preview}")
print(f"   ⏱️  {phase_5_time:.3f}s")
print()

# ===== PIPELINE SUMMARY =====
pipeline_total = time.time() - pipeline_start
print("=" * 70)
print("📊 RESUMO DO PIPELINE")
print("=" * 70)
print(f"   🏁 Tempo total: {pipeline_total:.3f}s")
print(f"   📋 Especificações: {len(pipeline_registry)}")
print(f"   🧪 Testes: {red_passed}/{len(red_results)} passaram")
print(f"   🤖 Agentes: {len(pipeline_orc.agents)}")
print(f"   ✅ Tarefas concluídas: {summary['completed']}/{summary['total_tasks']}")
print(f"   ⚠️  Falhas: {summary['failed']}")
print(f"   ⏱️  Execução: {summary['total_time']:.3f}s")
print(f"   📊 Média/tarefa: {summary['avg_time_per_task']:.3f}s")
print()
print("✅ Pipeline completo executado com sucesso!")


# 📊 Sumário Final

Resultados consolidados do pipeline completo de orquestração.


In [ ]:
#@title Sumário Final e Métricas de Performance
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich import box
    HAS_RICH = True
except ImportError:
    HAS_RICH = False

print("=" * 70)
print("📊 SUMÁRIO FINAL — ORQUESTRAÇÃO DE IA: SDD, TDD E MULTI-AGENTES")
print("=" * 70)
print()

if HAS_RICH:
    console = Console()

    # Main metrics
    main_table = Table(title="📊 Métricas do Pipeline", box=box.ROUNDED)
    main_table.add_column("Métrica", style="cyan", no_wrap=True)
    main_table.add_column("Valor", style="green", justify="right")

    main_table.add_row("Versão Python", sys.version[:5])
    main_table.add_row("Especificações SDD", str(len(pipeline_registry)))
    main_table.add_row("Testes TDD", f"{red_passed}/{len(red_results)}")
    main_table.add_row("Agentes Registrados", str(len(pipeline_orc.agents)))
    main_table.add_row("Tarefas Concluídas", str(summary['completed']))
    main_table.add_row("Tarefas com Falha", str(summary['failed']))
    main_table.add_row("Tempo Total Pipeline", f"{pipeline_total:.3f}s")
    main_table.add_row("Tempo Médio por Tarefa", f"{summary['avg_time_per_task']:.3f}s")

    console.print(main_table)
    print()

    # Per-agent table
    agent_table = Table(title="🤖 Performance por Agente", box=box.ROUNDED)
    agent_table.add_column("Agente", style="cyan")
    agent_table.add_column("Papel", style="magenta")
    agent_table.add_column("Execuções", style="green", justify="right")
    agent_table.add_column("Tempo Total", style="blue", justify="right")
    agent_table.add_column("Média", style="yellow", justify="right")

    for agent in pipeline_orc.list_agents():
        m = pipeline_orc.metrics[agent.name]
        agent_table.add_row(
            agent.name, agent.role,
            str(m["executions"]),
            f"{m['total_time']:.3f}s",
            f"{m['avg_time']:.3f}s",
        )

    console.print(agent_table)
    print()

    # Last logs
    log_text = "\n".join(pipeline_logging.logs[-6:])
    console.print(Panel(log_text, title="📋 Últimos Logs", border_style="blue"))
    print()

    # Tech stack
    tech_table = Table(title="🛠️ Tecnologias Utilizadas", box=box.SIMPLE)
    tech_table.add_column("Componente", style="cyan")
    tech_table.add_column("Descrição", style="white")
    tech_table.add_row("Pydantic v2", "Modelagem de especificações (SDD)")
    tech_table.add_row("TDD Engine", "Ciclo RED → GREEN → REFACTOR")
    tech_table.add_row("Orquestrador", "Registro → Roteamento → Execução")
    tech_table.add_row("Hook System", "Eventos: start, complete, error")
    tech_table.add_row("Prompt Patterns", "CoT, JSON, Few-shot, System")
    tech_table.add_row("Rich Library", "Visualização de dados tabulares")
    console.print(tech_table)

else:
    # Fallback without rich
    print("📊 Métricas do Pipeline:")
    print(f"   Python: {sys.version[:5]}")
    print(f"   SDD Specs: {len(pipeline_registry)}")
    print(f"   TDD Tests: {red_passed}/{len(red_results)}")
    print(f"   Agentes: {len(pipeline_orc.agents)}")
    print(f"   Tarefas: {summary['completed']} OK, {summary['failed']} falha")
    print(f"   Tempo: {pipeline_total:.3f}s total, "
          f"{summary['avg_time_per_task']:.3f}s médio/tarefa")
    print()
    print("Performance por Agente:")
    for agent in pipeline_orc.list_agents():
        m = pipeline_orc.metrics[agent.name]
        print(f"   {agent.name} ({agent.role}): {m['executions']} exec, "
              f"média {m['avg_time']:.3f}s")
    print()
    print("Últimos Logs:")
    for log in pipeline_logging.logs[-6:]:
        print(f"   {log}")

print()
print("=" * 70)
print("✅ Notebook executado com sucesso!")
print("🧠 Orquestração de IA: SDD, TDD e Multi-Agentes")
print("📅", datetime.now().strftime("%Y-%m-%d %H:%M"))
print("=" * 70)
